In [3]:
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# load models
model_lr = pickle.load(open("../models/model_lr.pkl", "rb"))
model_nb = pickle.load(open("../models/model_nb.pkl", "rb"))
vectorizer = pickle.load(open("../models/vectorizer.pkl", "rb"))
mlb = pickle.load(open("../models/mlb.pkl", "rb"))

# load processed dataset
df = pd.read_csv("../data/processed_dataset.csv")

# vectorize dataset
X_all = vectorizer.transform(df["summaries"])

In [5]:
def predict_labels(text):
    vec = vectorizer.transform([text])
    
    p1 = model_lr.predict_proba(vec)
    p2 = model_nb.predict_proba(vec)
    
    probs = (0.6 * p1 + 0.4 * p2)
    probs = np.power(probs, 1.2)
    
    idx = np.where(probs[0] >= 0.3)[0]
    
    if len(idx) == 0:
        idx = np.argsort(probs[0])[-1:]
    
    return [mlb.classes_[i] for i in idx]

In [15]:
def recommend_papers(query, top_k=5):
    
    # Step 1: predict labels
    labels = predict_labels(query)
    
    # Step 2: filter dataset
    filtered_df = df[df["terms"].apply(
        lambda x: any(label in x for label in labels)
    )]
    
    if len(filtered_df) == 0:
        filtered_df = df
    
    X_filtered = vectorizer.transform(filtered_df["summaries"])
    
    # Step 3: similarity
    query_vec = vectorizer.transform([query])
    sim = cosine_similarity(query_vec, X_filtered)[0]
    
    top_idx = sim.argsort()[-top_k:][::-1]
    
    return filtered_df.iloc[top_idx][[
        "titles", "authors", "terms", "published_date", "summaries"
    ]]

In [22]:
def smart_recommend(query, top_k=10):
    
    # 🔥 Step 1: Predict labels
    vec = vectorizer.transform([query])
    
    p1 = model_lr.predict_proba(vec)
    p2 = model_nb.predict_proba(vec)
    
    probs = (0.6 * p1 + 0.4 * p2)
    probs = np.power(probs, 1.2)
    
    idx = np.where(probs[0] >= 0.3)[0]
    
    if len(idx) == 0:
        idx = np.argsort(probs[0])[-1:]
    
    labels = [mlb.classes_[i] for i in idx]
    
    
    # 🔥 Step 2: Filter dataset
    filtered_df = df[df["terms"].apply(
        lambda x: any(label in x for label in labels)
    )]
    
    if len(filtered_df) == 0:
        filtered_df = df
    
    X_filtered = vectorizer.transform(filtered_df["summaries"])
    
    
    # 🔥 Step 3: Similarity
    sim = cosine_similarity(vec, X_filtered)[0]
    
    top_idx = sim.argsort()[-top_k:][::-1]
    
    
    # 🔥 Step 4: Return (UPDATED)
    return {
        "predicted_labels": labels,
        "recommendations": filtered_df.iloc[top_idx][[
            "titles", 
            "authors", 
            "terms", 
            "published_date",
            "summaries"   # ✅ ADDED
        ]]
    }

In [16]:
print(recommend_papers("Ai powered tourism"))
# print(predict_labels("deep learning for medical images"))

                                                  titles  \
13646  Empowering Local Communities Using Artificial ...   
20567  AI Liability Insurance With an Example in AI-P...   
8179                      German Tourism Knowledge Graph   
4218   Artificial Intelligence Systems applied to tou...   
4183   A Framework for Addressing the Risks and Oppor...   

                                                 authors      terms  \
13646  ['Yen-Chia Hsu', "Ting-Hao 'Kenneth' Huang", '...  ['cs.AI']   
20567                       ['Yunfei Ge', 'Quanyan Zhu']  ['cs.AI']   
8179   ['Umutcan Serles', 'Elias Kärle', 'Richard Hun...  ['cs.AI']   
4218   ['Luis Duarte', 'Jonathan Torres', 'Vitor Ribe...  ['cs.AI']   
4183   ['Sonia Baee', 'Mark Rucker', 'Anna Baglione',...  ['cs.AI']   

      published_date                                          summaries  
13646     10-05-2021  Artificial Intelligence (AI) is increasingly u...  
20567     06-01-2023  Artificial Intelligence (AI) has received 

In [23]:
# print(smart_recommend("Ai powered tourism"))
# print(smart_recommend("deep learning for medical images"))
print(smart_recommend("Smart traffic light system"))

{'predicted_labels': ['cs.LG'], 'recommendations':                                                   titles  \
35695  Deep Reinforcement Learning for the Joint Cont...   
96100  Deep Reinforcement Learning for Traffic Light ...   
27239  A Traffic Light Dynamic Control Algorithm with...   
78961  Deep Reinforcement Learning for Intelligent Tr...   
45847  Bait and Switch: Online Training Data Poisonin...   
25908  Traffic Light Control Using Deep Policy-Gradie...   
19082  Deep Reinforcement Learning for Traffic Light ...   
37803  Using Petri Nets as an Integrated Constraint M...   
36547  Traffic Pattern Classification in Smart Cities...   
35999  Amortized Network Intervention to Steer the Ex...   

                                                 authors      terms  \
35695  ['Johannes V. S. Busch', 'Robert Voelckner', '...  ['cs.LG']   
96100  ['Xiaoyuan Liang', 'Xunsheng Du', 'Guiling Wan...  ['cs.LG']   
27239     ['Xiaorong Hu', 'Chenguang Zhao', 'Gang Wang']  ['cs.LG']   
7896